In [ ]:
# --- Settings -------------------------------------------------------------
RUN = 'latest'  # 'A'..'E', or 'latest' = newest run for every MTU
DAY = 'today'   # 'today', 'tomorrow' or a delivery day like '2026-11-21'
#
# Run timing for delivery day D (local time):
#   IDCC(a)           calculated D-1 15:00, covers the whole of D
#   IDCC(b)           calculated D-1 22:00, covers the whole of D
#   IDCC(c)           calculated D 04:00, covers 06:00 onwards
#   IDCC(d)           calculated D 10:00, covers 12:00 onwards
#   IDCC(e)           calculated D 16:00, covers 18:00 onwards

# In Google Colab, first run:
#   !git clone https://github.com/lublazek/jao-atc-calculator.git
#   %cd jao-atc-calculator

import pandas as pd
from jao_atc.download import fetch_all, select

dfs = fetch_all(DAY)          # downloads all runs a-e, skips unpublished ones
df  = select(dfs, RUN)
if 'run' in df:               # 'latest': show which run each part of the day comes from
    print('\nlatest:', ', '.join(f'IDCC({r.lower()}) from {g.index.min():%H:%M}' for r, g in df.groupby('run', sort=False)))
    df = df.drop(columns='run')
else:
    print(f'\nShowing IDCC({RUN.lower()})')

In [ ]:
from jao_atc.capacity import all_pairs

COUNTRIES = ['CZ', 'SK', 'HU']            # zones to print (any of the 12 Core zones)

flow_15 = all_pairs(df).round(0)          # max-flow for every zone pair, MW
flow_h  = flow_15.resample('1h').min()    # hourly = min of the four quarter-hours
print(f'Max-flow ready: {len(flow_15)} MTUs x {flow_15.shape[1]} zone pairs')

In [ ]:
def show(flow, title):
    zones = sorted({z for col in flow.columns for z in col.split('>')})
    fmt = lambda v: 'N/A' if pd.isna(v) else str(int(v))
    print(f'\n{"=" * 90}\n  {title}\n{"=" * 90}')
    for ts, r in flow.iterrows():
        blocks = []
        for c in COUNTRIES:
            lines = [f'{c:<6}{"import":>8}  {"export":>8}', '-' * 24]
            lines += [f'{n:<6}{fmt(r[f"{n}>{c}"]):>8}  {fmt(r[f"{c}>{n}"]):>8}' for n in zones if n != c]
            blocks.append(lines)
        print(f'\n  {ts:%m-%d %H:%M}')
        for line in zip(*blocks):
            print('  ' + '  |  '.join(f'{part:<24}' for part in line))

In [ ]:
show(flow_15, '15-MINUTE CAPACITIES (max-flow)')

In [ ]:
show(flow_h, 'HOURLY CAPACITIES (min of 15-min within each hour)')